In [ ]:
# Gridmatic ERCOT POC - Battery Optimization Analysis
# This notebook implements battery dispatch optimization using CVXPY

import os
import pandas as pd
import numpy as np
from google.cloud import bigquery
import cvxpy as cp
import matplotlib.pyplot as plt

# Setup
PROJECT = os.environ["GCP_PROJECT"]
DATASET = os.environ.get("BQ_DATASET", "energy_ercot")
client = bigquery.Client(project=PROJECT)

print(f"🔍 Loading forecast data from {PROJECT}.{DATASET}")
print("⚡ Setting up battery optimization...")


In [ ]:
# Load forecast from BigQuery
fc = client.query(f"""
  SELECT timestamp_hour, yhat AS price
  FROM `{PROJECT}.{DATASET}.gold_forecasts`
  WHERE timestamp_hour >= TIMESTAMP_TRUNC(CURRENT_TIMESTAMP(), HOUR)
  ORDER BY timestamp_hour
""").to_dataframe()

p = fc["price"].to_numpy()
T = len(p)

print(f"✅ Loaded {T} forecast points")
print(f"📈 Price range: ${p.min():.2f} - ${p.max():.2f}")
print(f"📅 Forecast period: {fc['timestamp_hour'].min()} to {fc['timestamp_hour'].max()}")


In [ ]:
# Solve convex optimization problem
print("🔋 Setting up battery optimization problem...")

# Battery parameters
S_max, P_max, eta_c, eta_d, lam = 1.0, 1.0, 0.95, 0.95, 0.01

# Optimization variables
c = cp.Variable(T, nonneg=True)  # Charge power
d = cp.Variable(T, nonneg=True)  # Discharge power  
b = cp.Variable(T, nonneg=True)  # Bid power
s = cp.Variable(T+1)              # State of charge

# Constraints
cons = [s[0] == 0.5*S_max]  # Start at 50% SOC

for t in range(T):
    cons += [s[t+1] == s[t] + eta_c*c[t] - d[t]/eta_d,  # SOC dynamics
             c[t] <= P_max,                             # Charge limit
             d[t] <= P_max]                             # Discharge limit

cons += [s >= 0, s <= S_max]  # SOC bounds

# Objective: maximize profit
obj = cp.Maximize(p@d - p@c + p@b - lam*cp.sum_squares(b))

print("⚙️ Solving optimization problem...")
cp.Problem(obj, cons).solve(solver=cp.ECOS)

print(f"✅ Optimization status: {cp.Problem(obj, cons).status}")
print(f"💰 Optimal objective value: {cp.Problem(obj, cons).value:.2f}")


In [ ]:
# Prepare optimization results
out = fc.copy()
out["charge_mwh"] = np.maximum(c.value, 0)
out["discharge_mwh"] = np.maximum(d.value, 0)
out["soc_mwh"] = np.maximum(s.value[1:], 0)
out["bid_mwh"] = np.maximum(b.value, 0)
out["expected_profit"] = float((p*(out["discharge_mwh"]-out["charge_mwh"]+out["bid_mwh"])).sum())

print("📊 Optimization Results Summary:")
print(f"   - Total Charge: {out['charge_mwh'].sum():.2f} MWh")
print(f"   - Total Discharge: {out['discharge_mwh'].sum():.2f} MWh")
print(f"   - Max SOC: {out['soc_mwh'].max():.2f} MWh")
print(f"   - Min SOC: {out['soc_mwh'].min():.2f} MWh")
print(f"   - Expected Profit: ${out['expected_profit']:,.2f}")


In [ ]:
# Write results to gold_optimizer_outputs table
print("💾 Writing optimization results to BigQuery...")

client.load_table_from_dataframe(out, f"{PROJECT}.{DATASET}.gold_optimizer_outputs").result()

print("✅ Optimization results saved to gold_optimizer_outputs table")
print(f"📊 Saved {len(out)} optimization records")


In [ ]:
# PLOT #1 — Battery Optimization Graph (Main visualization)
plt.figure(figsize=(14, 8))

# Create subplots
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Price signal
ax1.plot(out['timestamp_hour'], out['price'], 
         color='blue', linewidth=2, label='Price Signal')
ax1.set_ylabel('Price ($/MWh)')
ax1.set_title('Battery Optimization Results - Price Signal & Dispatch', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Charge/Discharge operations
ax2.bar(out['timestamp_hour'], out['charge_mwh'], 
        color='green', alpha=0.7, label='Charge', width=0.8)
ax2.bar(out['timestamp_hour'], -out['discharge_mwh'], 
        color='red', alpha=0.7, label='Discharge', width=0.8)
ax2.set_ylabel('Power (MW)')
ax2.set_title('Charge/Discharge Operations')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Plot 3: State of Charge
ax3.plot(out['timestamp_hour'], out['soc_mwh'], 
         color='purple', linewidth=2, marker='o', markersize=4, label='SOC')
ax3.fill_between(out['timestamp_hour'], 0, out['soc_mwh'], 
                 alpha=0.3, color='purple')
ax3.set_ylabel('SOC (MWh)')
ax3.set_xlabel('Time')
ax3.set_title('Battery State of Charge')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, S_max * 1.1)

# Format x-axis
import matplotlib.dates as mdates
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax3.xaxis.set_major_locator(mdates.HourLocator(interval=6))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

print("🔋 Battery optimization visualization complete!")
print("💡 Key insights:")
print(f"   - Battery charges when prices are low")
print(f"   - Battery discharges when prices are high") 
print(f"   - SOC stays within bounds (0 to {S_max} MWh)")
print(f"   - Total profit: ${out['expected_profit']:,.2f}")


# Energy Dispatch Optimization

This notebook demonstrates optimization of energy dispatch using CVXPY to minimize costs while meeting demand forecasts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
# Generate sample demand forecast (would come from Prophet model)
def generate_demand_forecast():
    """Generate 24-hour demand forecast"""
    hours = np.arange(24)
    
    # Base demand pattern with daily cycle
    base_demand = 50000
    daily_pattern = 5000 * np.sin(2 * np.pi * hours / 24 - np.pi/2)  # Peak at 6 PM
    
    # Add some randomness
    noise = np.random.normal(0, 1000, 24)
    
    demand = base_demand + daily_pattern + noise
    
    return pd.DataFrame({
        'hour': hours,
        'demand': demand
    })

demand_forecast = generate_demand_forecast()
print("24-Hour Demand Forecast:")
print(demand_forecast.head())

# Plot demand forecast
plt.figure(figsize=(12, 6))
plt.plot(demand_forecast['hour'], demand_forecast['demand'], marker='o', linewidth=2)
plt.title('24-Hour Demand Forecast')
plt.xlabel('Hour of Day')
plt.ylabel('Demand (MW)')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Define optimization problem
def optimize_dispatch(demand_forecast):
    """Optimize energy dispatch to minimize cost"""
    
    n_hours = len(demand_forecast)
    demand = demand_forecast['demand'].values
    
    # Decision variables
    solar = cp.Variable(n_hours, nonneg=True)
    wind = cp.Variable(n_hours, nonneg=True)
    gas = cp.Variable(n_hours, nonneg=True)
    battery_charge = cp.Variable(n_hours, nonneg=True)
    battery_discharge = cp.Variable(n_hours, nonneg=True)
    battery_soc = cp.Variable(n_hours + 1, nonneg=True)
    
    # Constraints
    constraints = []
    
    # Power balance constraint
    for i in range(n_hours):
        constraints.append(
            solar[i] + wind[i] + gas[i] + battery_discharge[i] - battery_charge[i] >= demand[i]
        )
    
    # Battery state of charge dynamics
    constraints.append(battery_soc[0] == 0.5)  # Initial SOC (50%)
    for i in range(n_hours):
        constraints.append(
            battery_soc[i+1] == battery_soc[i] + 0.9 * battery_charge[i] - battery_discharge[i]
        )
        constraints.append(battery_soc[i+1] <= 1.0)  # Max SOC (100%)
        constraints.append(battery_soc[i+1] >= 0.1)  # Min SOC (10%)
    
    # Generation capacity constraints
    constraints.append(solar <= 500)  # Max solar capacity (MW)
    constraints.append(wind <= 300)   # Max wind capacity (MW)
    constraints.append(gas <= 800)   # Max gas capacity (MW)
    
    # Battery power constraints
    constraints.append(battery_charge <= 200)   # Max charge rate (MW)
    constraints.append(battery_discharge <= 200) # Max discharge rate (MW)
    
    # Objective: minimize total cost
    solar_cost = 0.05  # $/MWh
    wind_cost = 0.03   # $/MWh
    gas_cost = 0.08    # $/MWh
    
    total_cost = cp.sum(solar_cost * solar + wind_cost * wind + gas_cost * gas)
    
    # Solve optimization problem
    problem = cp.Problem(cp.Minimize(total_cost), constraints)
    problem.solve()
    
    if problem.status == cp.OPTIMAL:
        return {
            'solar': solar.value,
            'wind': wind.value,
            'gas': gas.value,
            'battery_charge': battery_charge.value,
            'battery_discharge': battery_discharge.value,
            'battery_soc': battery_soc.value[1:],
            'total_cost': problem.value,
            'status': 'optimal'
        }
    else:
        return {'status': 'infeasible', 'total_cost': float('inf')}

# Run optimization
result = optimize_dispatch(demand_forecast)

if result['status'] == 'optimal':
    print(f"Optimization successful!")
    print(f"Total cost: ${result['total_cost']:.2f}")
else:
    print("Optimization failed - problem is infeasible")


In [ ]:
# Visualize optimization results
if result['status'] == 'optimal':
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    hours = demand_forecast['hour']
    
    # Generation mix
    axes[0, 0].stackplot(hours, 
                        result['solar'], 
                        result['wind'], 
                        result['gas'],
                        labels=['Solar', 'Wind', 'Gas'],
                        alpha=0.7)
    axes[0, 0].plot(hours, demand_forecast['demand'], 'k-', linewidth=2, label='Demand')
    axes[0, 0].set_title('Generation Mix vs Demand')
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Power (MW)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Battery operations
    axes[0, 1].plot(hours, result['battery_charge'], 'g-', label='Charge', linewidth=2)
    axes[0, 1].plot(hours, result['battery_discharge'], 'r-', label='Discharge', linewidth=2)
    axes[0, 1].set_title('Battery Operations')
    axes[0, 1].set_xlabel('Hour of Day')
    axes[0, 1].set_ylabel('Power (MW)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Battery SOC
    axes[1, 0].plot(hours, result['battery_soc'], 'b-', linewidth=2)
    axes[1, 0].set_title('Battery State of Charge')
    axes[1, 0].set_xlabel('Hour of Day')
    axes[1, 0].set_ylabel('SOC (%)')
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Cost breakdown
    solar_cost_total = np.sum(result['solar']) * 0.05
    wind_cost_total = np.sum(result['wind']) * 0.03
    gas_cost_total = np.sum(result['gas']) * 0.08
    
    costs = [solar_cost_total, wind_cost_total, gas_cost_total]
    labels = ['Solar', 'Wind', 'Gas']
    colors = ['yellow', 'lightblue', 'orange']
    
    axes[1, 1].pie(costs, labels=labels, colors=colors, autopct='%1.1f%%')
    axes[1, 1].set_title('Cost Breakdown')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nOptimization Summary:")
    print(f"Total Solar Generation: {np.sum(result['solar']):.1f} MWh")
    print(f"Total Wind Generation: {np.sum(result['wind']):.1f} MWh")
    print(f"Total Gas Generation: {np.sum(result['gas']):.1f} MWh")
    print(f"Peak Battery SOC: {np.max(result['battery_soc'])*100:.1f}%")
    print(f"Min Battery SOC: {np.min(result['battery_soc'])*100:.1f}%")
